In [ ]:
import numpy as np
import pandas as pd

input_file = "emnist-digits-train.csv"
df_train = pd.read_csv(input_file, header=None)

In [ ]:
data_train = df_train.to_numpy()
data_train.shape

In [ ]:
X_train = data_train[:, 1:].astype(np.float32)
y_train = data_train[:, 0].copy()
del df_train, data_train
X_train.shape, y_train.shape

In [ ]:
# 특성 엔지니어링 (HOG 미사용): 모든 특성은 이미지 한 장만 보고 계산 (다른 행·전체 데이터 통계 사용 안 함)
#  deskew: 기울어진 글씨를 똑바로 세움 / stroke_features: 획 교차 횟수·위/왼쪽 잉크 비율·구멍 / profiles: 상하좌우 윤곽 거리
from scipy import ndimage

def deskew(imgs):
    out = np.empty_like(imgs)
    yy, xx = np.mgrid[:28, :28].astype(np.float32)
    for i, im in enumerate(imgs):
        tot = im.sum() + 1e-6
        cy, cx = (yy * im).sum() / tot, (xx * im).sum() / tot
        vy = ((yy - cy) ** 2 * im).sum() / tot
        cov = ((yy - cy) * (xx - cx) * im).sum() / tot
        alpha = cov / (vy + 1e-6)
        mat = np.array([[1, 0], [alpha, 1]], dtype=np.float32)
        offset = np.array([cy, cx], dtype=np.float32) - mat @ np.array([14, 14], dtype=np.float32)
        out[i] = ndimage.affine_transform(im, mat, offset=offset, order=1)
    return out

def holes(imgs, th=0.3):
    out = np.zeros((len(imgs), 2), dtype=np.float32)
    for i, im in enumerate(imgs):
        lab, n = ndimage.label(im < th)
        border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])))
        inner = [k for k in range(1, n + 1) if k not in border]
        out[i, 0] = len(inner)
        out[i, 1] = sum((lab == k).sum() for k in inner)
    return out

def simple(X):
    imgs = X.reshape(-1, 28, 28)
    return np.hstack([imgs.sum(2), imgs.sum(1), imgs.reshape(-1, 7, 4, 7, 4).mean(axis=(2, 4)).reshape(-1, 49),
                      (imgs > 0.1).sum(axis=(1, 2))[:, None].astype(np.float32)])

def stroke_features(dk):
    b = (dk > 0.3).astype(np.int8)
    tr_r = (np.diff(b, axis=2) == 1).sum(2) / 4.0
    tr_c = (np.diff(b, axis=1) == 1).sum(1) / 4.0
    ink = dk.sum((1, 2)).clip(1e-6)
    top = dk[:, :10, :].sum((1, 2)) / ink
    left = dk[:, :, :14].sum((1, 2)) / ink
    h = holes(dk) / np.array([1, 50], np.float32)
    return np.hstack([tr_r, tr_c, top[:, None], left[:, None], h]).astype(np.float32)

def profiles(dk):
    b = dk > 0.3
    def first(m, axis):
        return np.where(m.any(axis), np.argmax(m, axis), 28) / 28.0
    left = first(b, 2)
    right = 1 - first(b[:, :, ::-1], 2)
    top = first(b, 1)
    bottom = 1 - first(b[:, ::-1, :], 1)
    return np.hstack([left, right, top, bottom]).astype(np.float32)

In [ ]:
def make_features(X_raw, chunk=10000):
    out = []
    for s in range(0, len(X_raw), chunk):
        X = X_raw[s:s + chunk] / 255.0
        dk = deskew(np.ascontiguousarray(X.reshape(-1, 28, 28).transpose(0, 2, 1)))
        out.append(np.hstack([dk.reshape(len(dk), -1), stroke_features(dk) * 1.5, profiles(dk)]).astype(np.float64))
    return np.vstack(out)

X_fe = make_features(X_train)
X_fe.shape

In [ ]:
# 1) 모델 비교: 원본 픽셀 vs 내 특성, 여러 분류 모델 (훈련셋 중 무작위 20,000개, 3-fold shuffle=True)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

kf = KFold(n_splits=10, shuffle=True, random_state=42)
kf3 = KFold(n_splits=3, shuffle=True, random_state=42)

sub = np.random.RandomState(0).permutation(len(X_train))[:20000]
Xs_raw, Xs, ys = X_train[sub] / 255.0, X_fe[sub], y_train[sub]
del X_train   # 메모리 절약 (원본 픽셀은 이후 사용하지 않음)

models = {
    "SGD": SGDClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVC": SVC(C=10),
}
for name, model in models.items():
    raw = cross_val_score(model, Xs_raw, ys, cv=kf3, scoring="accuracy", n_jobs=-1).mean()
    fe = cross_val_score(model, Xs, ys, cv=kf3, scoring="accuracy", n_jobs=-1).mean()
    print(f"{name:20s} 원본 {raw:.4f}   내 특성 {fe:.4f}")

In [ ]:
# 2) 하이퍼파라미터 탐색: 가장 좋은 SVC 의 C, gamma (Grid Search, 20,000개, 3-fold shuffle=True)
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [10, 30], "gamma": [0.01, 0.015, 0.02, 0.025]}
grid_search = GridSearchCV(SVC(cache_size=1000), param_grid, cv=kf3, scoring="accuracy", n_jobs=-1)
grid_search.fit(Xs, ys)
print(grid_search.best_params_, grid_search.best_score_)
for mean, params in zip(grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["params"]):
    print(f"{mean:.4f}", params)

In [ ]:
# 3) 최종 모델: 전체 240,000개 10-fold 교차 검증 (shuffle=True)
#  cross_val_predict 로 fold 별 예측을 한 번에 구하고, 같은 fold 로 정확도와 오차 행렬을 계산
from sklearn.model_selection import cross_val_predict

final_clf = SVC(cache_size=2000, **grid_search.best_params_)
y_pred = cross_val_predict(final_clf, X_fe, y_train, cv=kf, n_jobs=2, verbose=3)
cv_res = np.array([(y_pred[te] == y_train[te]).mean() for _, te in kf.split(X_fe)])
cv_res

In [ ]:
np.mean(cv_res)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_train, y_pred)